# 📖 Notebook 3: Replication & Failover

A single PostgreSQL server is a **single point of failure**. If it goes down, your entire application goes down. Replication solves this by keeping copies of your data on multiple servers.

## Learning Objectives

By the end of this notebook, you'll understand:
- How streaming replication works in PostgreSQL
- How to read from replicas to scale reads
- How to monitor replication lag
- What happens during failover

## The Pattern: BAD → BETTER → BEST

| Approach | Setup | Risk |
|----------|-------|------|
| 🔴 BAD | Single server, no replication | Total downtime if server fails |
| 🟡 BETTER | Primary + replica (manual failover) | Reads scale, but failover is slow |
| 🟢 BEST | Primary + replica + monitoring + auto-failover | High availability |

## 🛠️ Setup

This lab's `docker-compose.yml` already runs a **primary** and a **replica**:

```
Primary (pg-primary) ──WAL──▶ Replica (pg-replica)
    port 5432                     port 5433
    read + write                  read-only
```

```bash
cd deep-dives/postgres
docker-compose up -d
# Wait ~30 seconds for the replica to sync
```

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import psycopg2
import time
from tabulate import tabulate

PRIMARY_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "postgres_demo",
    "user": "demo",
    "password": "demo"
}

REPLICA_CONFIG = {
    "host": "localhost",
    "port": 5433,
    "database": "postgres_demo",
    "user": "demo",
    "password": "demo"
}

def get_primary():
    return psycopg2.connect(**PRIMARY_CONFIG)

def get_replica():
    return psycopg2.connect(**REPLICA_CONFIG)

def run_on(conn_fn, sql, params=None):
    conn = conn_fn()
    conn.autocommit = True
    cur = conn.cursor()
    cur.execute(sql, params)
    result = cur.fetchall() if cur.description else None
    cols = [d[0] for d in cur.description] if cur.description else []
    conn.close()
    return result, cols

# Test both connections
try:
    conn = get_primary()
    cur = conn.cursor()
    cur.execute("SELECT pg_is_in_recovery()")
    is_replica = cur.fetchone()[0]
    conn.close()
    print(f"✅ Primary (port 5432): connected — is_replica={is_replica}")
except Exception as e:
    print(f"❌ Primary failed: {e}")

try:
    conn = get_replica()
    cur = conn.cursor()
    cur.execute("SELECT pg_is_in_recovery()")
    is_replica = cur.fetchone()[0]
    conn.close()
    print(f"✅ Replica (port 5433): connected — is_replica={is_replica}")
except Exception as e:
    print(f"❌ Replica failed: {e}")
    print("   The replica may need ~30 seconds to sync after first startup.")
    print("   Run: docker-compose up -d && sleep 30")

---

## 🔴 BAD: Single Server (No Replication)

With a single PostgreSQL server:
- **All reads AND writes** go to one server
- If that server crashes, **everything stops**
- As traffic grows, the single server becomes a bottleneck

```
    ┌──────────────────┐
    │   App Server(s)   │
    └────────┬─────────┘
             │ ALL reads + writes
             ▼
    ┌──────────────────┐
    │   PostgreSQL      │ ◀── Single point of failure!
    │   (single server) │
    └──────────────────┘
```

Let's see the problem: ALL traffic hits the primary.

In [ ]:
# 🔴 BAD: All traffic hits the single primary
# Simulate 100 read requests all going to the primary

print("=" * 60)
print("🔴 BAD: All reads go to PRIMARY (single server)")
print("=" * 60)
print()

times = []
for i in range(100):
    start = time.time()
    run_on(get_primary, "SELECT * FROM posts WHERE user_id = %s", ((i % 5000) + 1,))
    times.append((time.time() - start) * 1000)

avg_primary_only = sum(times) / len(times)
print(f"⏱️  100 reads on PRIMARY only:")
print(f"   Average: {avg_primary_only:.2f} ms per query")
print(f"   Total:   {sum(times):.0f} ms")
print()
print("💡 Under high concurrency, this single server would")
print("   become a bottleneck and eventually crash under load.")

## 🟡 BETTER: Primary + Replica (Read Scaling)

With streaming replication, the replica gets a **real-time copy** of all data from the primary. We can send **read queries to the replica**, reducing load on the primary.

```
    ┌──────────────────┐
    │   App Server(s)   │
    └───┬──────────┬───┘
        │ writes   │ reads
        ▼          ▼
    ┌────────┐  ┌────────┐
    │ PRIMARY │─▶│ REPLICA│
    │ (5432)  │  │ (5433) │
    └────────┘  └────────┘
       WAL streaming
```

In [ ]:
# 🟡 BETTER: Split reads to replica, writes to primary

print("=" * 60)
print("🟡 BETTER: Read from REPLICA (port 5433)")
print("=" * 60)
print()

# First, verify the replica has the same data
primary_count, _ = run_on(get_primary, "SELECT COUNT(*) FROM posts")
replica_count, _ = run_on(get_replica, "SELECT COUNT(*) FROM posts")

print(f"📊 Data comparison:")
print(f"   Primary: {primary_count[0][0]:,} posts")
print(f"   Replica: {replica_count[0][0]:,} posts")
print(f"   In sync: {'✅ Yes' if primary_count == replica_count else '⚠️ Lag detected'}")
print()

# Now benchmark reads from the replica
times_replica = []
for i in range(100):
    start = time.time()
    run_on(get_replica, "SELECT * FROM posts WHERE user_id = %s", ((i % 5000) + 1,))
    times_replica.append((time.time() - start) * 1000)

avg_replica = sum(times_replica) / len(times_replica)
print(f"⏱️  100 reads on REPLICA:")
print(f"   Average: {avg_replica:.2f} ms per query")
print()
print("💡 The replica handles reads just as fast as the primary.")
print("   In production, you'd have multiple replicas to spread load.")

In [ ]:
# Verify: the replica is READ-ONLY (writes should fail)

print("🔒 Verifying replica is read-only...")
print()

try:
    conn = get_replica()
    conn.autocommit = True
    cur = conn.cursor()
    cur.execute("INSERT INTO users (username, email) VALUES ('test_write', 'test@test.com')")
    conn.close()
    print("❌ Write SUCCEEDED — something is wrong, replica should be read-only!")
except Exception as e:
    print(f"✅ Write correctly REJECTED on replica:")
    print(f"   Error: {e}")
    print()
    print("💡 Replicas are strictly read-only.")
    print("   All writes MUST go to the primary.")

---

## How Streaming Replication Works

```
  PRIMARY                                    REPLICA
  ┌──────────────────┐                     ┌──────────────────┐
  │ 1. Client writes  │                     │                  │
  │    INSERT INTO ... │                     │                  │
  │                    │                     │                  │
  │ 2. Write to WAL    │                     │                  │
  │    (Write-Ahead    │──── WAL stream ────▶│ 4. Apply WAL     │
  │     Log)           │                     │    changes to    │
  │                    │                     │    local data    │
  │ 3. Apply to tables │                     │                  │
  └──────────────────┘                     └──────────────────┘
```

**WAL (Write-Ahead Log)**: Every change is first written to a log file. The replica reads this log in real-time and applies the same changes. This is called **streaming replication**.

In [ ]:
# Let's watch replication in real-time!
# We'll write to the primary and see it appear on the replica.

print("=" * 60)
print("👀 Watching Replication in Real-Time")
print("=" * 60)
print()

# Step 1: Check current state
before_primary, _ = run_on(get_primary, "SELECT COUNT(*) FROM posts")
before_replica, _ = run_on(get_replica, "SELECT COUNT(*) FROM posts")
print(f"Before: Primary={before_primary[0][0]:,} posts, Replica={before_replica[0][0]:,} posts")

# Step 2: Write new data to PRIMARY
conn = get_primary()
conn.autocommit = True
cur = conn.cursor()
cur.execute(
    "INSERT INTO posts (user_id, title, content, status) VALUES (%s, %s, %s, %s) RETURNING id",
    (1, 'Replication test post', 'This was written to the primary!', 'published')
)
new_id = cur.fetchone()[0]
conn.close()
print(f"\nWritten: Post #{new_id} to PRIMARY")

# Step 3: Read from REPLICA (should appear almost instantly)
time.sleep(0.5)  # tiny wait for WAL to stream

after_replica, _ = run_on(get_replica, "SELECT COUNT(*) FROM posts")
replica_post, _ = run_on(get_replica, "SELECT title FROM posts WHERE id = %s", (new_id,))

print(f"\nAfter:  Replica={after_replica[0][0]:,} posts")
if replica_post:
    print(f"Read from replica: '{replica_post[0][0]}'")
    print(f"\n✅ Data replicated in < 500ms!")
else:
    print("⚠️ Data not yet on replica — replication lag detected")

# Clean up the test post
conn = get_primary()
conn.autocommit = True
cur = conn.cursor()
cur.execute("DELETE FROM posts WHERE id = %s", (new_id,))
conn.close()

---

## 🟢 BEST: Monitoring Replication Lag

In production, you **must** monitor replication lag. If the replica falls behind, reads from it return **stale data**.

In [ ]:
# Check replication status on the primary

print("=" * 60)
print("🟢 BEST: Monitor Replication Health")
print("=" * 60)
print()

# Query the primary for replication stats
repl_stats, cols = run_on(get_primary, """
    SELECT
        client_addr,
        state,
        sent_lsn,
        write_lsn,
        flush_lsn,
        replay_lsn,
        pg_wal_lsn_diff(sent_lsn, replay_lsn) AS replication_lag_bytes
    FROM pg_stat_replication
""")

if repl_stats:
    print("📡 Replication Status (from PRIMARY):")
    print(tabulate(repl_stats, headers=cols, tablefmt="simple_grid"))
    print()
    lag_bytes = repl_stats[0][6] if repl_stats[0][6] else 0
    print(f"📏 Replication lag: {lag_bytes} bytes")
    if lag_bytes < 1024:
        print("✅ Lag is minimal — replica is nearly real-time")
    else:
        print(f"⚠️ Lag is {lag_bytes / 1024:.1f} KB — replica is falling behind")
else:
    print("⚠️ No replication connections found")
    print("   Make sure the replica is running: docker-compose up -d")

print()

# Check from the replica's perspective
replica_info, _ = run_on(get_replica, """
    SELECT
        pg_is_in_recovery() AS is_replica,
        pg_last_wal_receive_lsn() AS last_received,
        pg_last_wal_replay_lsn() AS last_replayed,
        pg_last_xact_replay_timestamp() AS last_replay_time
""")
if replica_info:
    print("📡 Replica Self-Report:")
    print(f"   Is replica:        {replica_info[0][0]}")
    print(f"   Last WAL received: {replica_info[0][1]}")
    print(f"   Last WAL replayed: {replica_info[0][2]}")
    print(f"   Last replay time:  {replica_info[0][3]}")

In [ ]:
# Simulate a read-scaling architecture
# Route writes to primary, reads to replica

import random

print("=" * 60)
print("🏗️ Simulating Read-Scaling Architecture")
print("=" * 60)
print()

def get_connection(operation="read"):
    """Route reads to replica, writes to primary."""
    if operation == "write":
        return get_primary()
    else:
        return get_replica()

# Simulate a workload: 90% reads, 10% writes
total_reads = 0
total_writes = 0
read_times = []
write_times = []

for i in range(200):
    if random.random() < 0.9:
        # READ operation — goes to replica
        start = time.time()
        conn = get_connection("read")
        conn.autocommit = True
        cur = conn.cursor()
        cur.execute("SELECT * FROM posts WHERE user_id = %s LIMIT 5", ((i % 5000) + 1,))
        cur.fetchall()
        conn.close()
        read_times.append((time.time() - start) * 1000)
        total_reads += 1
    else:
        # WRITE operation — goes to primary
        start = time.time()
        conn = get_connection("write")
        conn.autocommit = True
        cur = conn.cursor()
        cur.execute(
            "UPDATE posts SET view_count = view_count + 1 WHERE id = %s",
            ((i % 100000) + 1,)
        )
        conn.close()
        write_times.append((time.time() - start) * 1000)
        total_writes += 1

print(f"📊 Workload Results (200 operations):")
print(f"   Reads (→ replica):  {total_reads} ops, avg {sum(read_times)/len(read_times):.2f} ms")
print(f"   Writes (→ primary): {total_writes} ops, avg {sum(write_times)/len(write_times):.2f} ms")
print()
print("💡 In production with 3 replicas, each handles ~30% of reads.")
print("   The primary only handles writes = much less load.")

## ⚠️ Failover: What Happens When the Primary Dies?

When the primary goes down:
1. The replica has a copy of all data (up to replication lag)
2. You **promote** the replica to become the new primary
3. Application connections switch to the new primary

```
BEFORE:                          AFTER FAILOVER:
┌────────┐    ┌────────┐        ┌────────┐
│ PRIMARY │───▶│ REPLICA│        │ NEW     │
│ (5432)  │    │ (5433) │   ──▶  │ PRIMARY │
└────────┘    └────────┘        │ (5433)  │
    💥 CRASH                    └────────┘
```

In this Docker lab, you can simulate this:

```bash
# Stop the primary
docker stop pg-primary

# The replica is still running with all data!
# In production, you would promote it with:
#   pg_ctl promote -D /var/lib/postgresql/data
#
# Or in modern PostgreSQL:
#   SELECT pg_promote();
```

**Note**: In production, tools like **Patroni**, **pg_auto_failover**, or cloud-managed
PostgreSQL (AWS RDS, Google Cloud SQL) handle automatic failover for you.

In [ ]:
# Let's check what data the replica has — it should be a full copy

print("=" * 60)
print("📋 Replica Data Verification")
print("=" * 60)
print()

tables = ['users', 'posts', 'comments', 'follows', 'likes', 'direct_messages']
results = []

for table in tables:
    primary_count, _ = run_on(get_primary, f"SELECT COUNT(*) FROM {table}")
    replica_count, _ = run_on(get_replica, f"SELECT COUNT(*) FROM {table}")
    p = primary_count[0][0]
    r = replica_count[0][0]
    match = "✅" if p == r else "⚠️"
    results.append((table, f"{p:,}", f"{r:,}", match))

print(tabulate(results,
               headers=["Table", "Primary", "Replica", "Match"],
               tablefmt="simple_grid"))
print()
print("💡 The replica is a byte-for-byte copy of the primary.")
print("   If the primary fails, the replica has ALL the data.")

## 📚 Summary

### Key Takeaways

1. **Streaming replication** sends WAL changes in real-time to replicas
2. **Replicas are read-only** — use them to scale reads, not writes
3. **Replication lag** is usually milliseconds, but monitor it!
4. **Failover** promotes a replica to primary — use Patroni or cloud tools for automation
5. **Route traffic**: writes → primary, reads → replica(s)

### Replication in System Design Interviews

| Question | Answer |
|----------|--------|
| How to scale reads? | Add read replicas |
| How to handle primary failure? | Promote replica, use auto-failover |
| What about replication lag? | Monitor it; use primary for critical reads |
| How many replicas? | 2-3 for most systems; more for read-heavy loads |

### Next Up

In **Notebook 4**, we'll explore **partitioning** — how to handle tables with billions of rows.